# Analyzing Text Responses from TAPE data

Steps:
* Load data
* Get length count and visualize
* Text cleaning - split at white space, remove punctuation, remove stop words
* Word cloud
* LDA topic allocation?



In [ ]:
# Installing all dependencies up front

# Pandas, numpy and OS
import pandas as pd
import numpy as np
import os

# Math and plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Text ckeanubg
import string
import re

# Word cloud and text cleaning
import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from wordcloud import WordCloud
import matplotlib.pyplot as plt


# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# PCA and tSNE
from sklearn.decomposition import TruncatedSVD

# Loading data

In [ ]:
# Load data

# df = pd.read_csv("/content/example.csv",) # Original line
quant = pd.read_csv("/Users/natajahroberts/Desktop/TAPE_GIPT_clean_merge(124).csv")
qual = pd.read_csv("/Users/natajahroberts/Desktop/TAPE_qual_w_demog.csv")

In [ ]:
quant.columns

In [ ]:
qual.columns

In [ ]:
# Remove extra rows on quant
quant = quant[1:122]

In [ ]:
# Remove duplicate rows on qual
qual = # just keep subject and 4 questions


In [ ]:
# Rename ID as Subject on quant
quant = quant.rename('Subject' : 'X.1')

In [ ]:
# Merge on Subject
full = pd.merge(quant, qual, on = 'Subject')  # check syntax

In [ ]:
# Checking results
full.columns


In [ ]:
# Creating subsets for visualization



# Basic EDA

In [ ]:
# Get length and visualize

review_length = reviews.str.len()

plt.figure(figsize=(8,5))
sns.histplot(review_length, bins=50, kde=True)
plt.title("Distribution of REVIEW response lengths")
plt.xlabel("Length of Response (characters)")
plt.ylabel("Frequency")
plt.xlim(0, 550)
plt.show()

In [ ]:
# Text cleaning

def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.lower()  # lowercase
    #text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text


reviews_clean = reviews.apply(clean_text)
reviews_clean[5]

In [ ]:
final = reviews_clean.str.cat()


## EDA for TAPE



In [ ]:
# Get length and visualize

df['Consequences_length'] = df['Consequences'].str.len()

plt.figure(figsize=(8,5))
sns.histplot(df['Consequences_length'], bins=50, kde=True)
plt.title("Distribution of Consequences response lengths")
plt.xlabel("Length of Response (characters)")
plt.ylabel("Frequency")
plt.show()

In [ ]:

df['Patients_length'] = df['Patients'].str.len()

plt.figure(figsize=(8,5))
sns.histplot(df['Patients_length'], bins=50, kde=True)
plt.title("Distribution of Patients response lengths")
plt.xlabel("Length of Response (characters)")
plt.ylabel("Frequency")
plt.show()

# NLTK Word Cloud demo

In [ ]:

#Function to generate a word cloud from user input text
def generate_wordcloud(text):
    words = word_tokenize(text)

    #Let's check the word count
    word_count = len(words)
    min_word_count = 2 #Setting minimum word count to be 25
    if word_count < min_word_count:
        print(f"Insufficient words. Please type at least {min_word_count} words.")
        return

    #Removing stopwords
    stop_words = set(stopwords.words('english'))
    words = [word.lower()
            for word in words
                if word.isalnum() and word.lower() not in stop_words]
    #Creating a word frequency
    word_freq = {}
    for word in words:
        if word in word_freq:
            word_freq[word] += 1
        else:
            word_freq[word] = 1
    #Generating the word cloud
    wordcloud = WordCloud (width=800, height=400, background_color='white').generate_from_frequencies(word_freq)

    #Displaying the word cloud
    plt.figure(figsize=(10, 5)) #width and height
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.show()

# SPLITTING DATA

Before we move on to TF-IDF, here's how to split the data set based on condition. Word clouds to show how it worked since the wordcloud function is already loaded.

In [ ]:
# Practicing splitting data set based on other variables
# Let's see what's in the sample data
df.columns


In [ ]:
# Need to see what a row looks like
df[0:5]

In [ ]:
# great, age is an easy criterai for subsetting
# so is rating, let's do both
age_mask = df['Age'] > np.mean(df['Age'])
older, younger = df[age_mask], df[~age_mask]



In [ ]:
# checking
younger[0:5]

In [ ]:
# How to access a column mean, put it within the mask tho
np.mean(df['Age'])

In [ ]:
# Now for rating
satis_mask = df['Rating'] > np.mean(df['Rating'])
happy, unhappy = df[satis_mask], df[~satis_mask]

In [ ]:
# Now we need to extract the text only and collapse it into a string
older_reviews, younger_reviews, happy_reviews, unhappy_reviews = older['Review Text'], younger['Review Text'], happy['Review Text'], unhappy['Review Text']


In [ ]:
type(happy_reviews)
# these are Pandas series, so we can't use subsetting to check the contents
# happy_reviews runs, but happy_reviews[1] won't

In [ ]:
# Collapsing series to strings

older_all = older_reviews.str.cat()
younger_all = younger_reviews.str.cat()
happy_all = happy_reviews.str.cat()
unhappy_all = unhappy_reviews.str.cat()

In [ ]:
generate_wordcloud(happy_all)

In [ ]:
generate_wordcloud(final)

#TF-IDF





In [ ]:
### Running TF-IDF with scikitlearn
## DO NOT EDIT  -- Don't need to run either ###

from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [sentence_a, sentence_b]
titles = ["seneca", "steve_jobs"]

vectorizer = TfidfVectorizer()
vector = vectorizer.fit_transform(corpus)
dict(zip(vectorizer.get_feature_names_out(), vector.toarray()[0]))

tfidf_df = pd.DataFrame(
    vector.toarray(), index=titles, columns=vectorizer.get_feature_names_out()
)

In [ ]:
# Original stop words
original_stops = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between", "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will", "just", "don", "should", "now"]

practice_stops = ["dress", "size", "fit"]
TAPE_stops = ["patient", "client", "care"]

type(original_stops)

In [ ]:
# Creating first custom stop list with list concatenation
first_custom = original_stops + practice_stops

In [ ]:
## Initializing vectorizer with custom list

vectorizer = TfidfVectorizer(stop_words=first_custom) # Pass the list to stop_words

# running vectorizer on reviews_clean cause we need separate documents
X_tfidf_custom = vectorizer.fit_transform(reviews_clean)

print(vectorizer.get_feature_names_out())


In [ ]:
# Testing multiple documents of sample data to explore viz options

corpus = [happy_all, unhappy_all, younger_all, older_all]
titles = ["happy reviews", "unhappy reviews", "young reviews", "old reviews"]

# Using the vectorizer
review_tfidf = vectorizer.fit_transform(corpus)

# Zipping the feature names with the scores
# dict(zip(vectorizer.get_feature_names_out(), review_tfidf.toarray()[0]))

# this just prints out everything. code below gives you a df without this step


In [ ]:
# Creating dataframe of results

tfidf_1 = pd.DataFrame(
    review_tfidf.toarray(), index=titles, columns=vectorizer.get_feature_names_out()
)
tfidf_1

In [ ]:
# Trying to inspect specific words

tfidf_1["small"]

# great!
# we see higher scores among older and unhappy reviews and lower among younger and happy

In [ ]:
# split these into 4 tf_idf sets for visualization

happy_tf = tfidf_1.iloc[0]
unhappy_tf = tfidf_1.iloc[1]
younger_tf = tfidf_1.iloc[2]
older_tf = tfidf_1.iloc[3]



### How to visualize TF-IDF

In [ ]:
# from medium article

# Visualization function
def visualize_tfidf(tfidf_matrix: pd.DataFrame):
    plt.figure(figsize=(10, 10))
    sns.heatmap(tfidf_matrix, annot=True, cmap="YlGnBu")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# Prepare the TF-IDF matrix for visualization and EDA
tfidf_matrix = pd.DataFrame([happy_tf, unhappy_tf, younger_tf, older_tf], index=["Happy", "Unhappy", "Younger", "Older"])

# Get the top 10 words for each group
top_words_per_group = {}
for index, row in tfidf_matrix.iterrows():
    top_words = row.nlargest(10).index.tolist()
    top_words_per_group[index] = top_words

# Flatten the list of top words and get unique words
all_top_words = list(set([word for sublist in top_words_per_group.values() for word in sublist]))

# Filter the tfidf_matrix to include only these top words
tfidf_matrix_top10 = tfidf_matrix[all_top_words]

# Visualize the TF-IDF matrix
visualize_tfidf(tfidf_matrix_top10)

In [ ]:
# This is an alternative visualization function for when the heatmap is squished and needs rotated annotations and labels
# Visualization function
def visualize_tfidf_rotated(tfidf_matrix: pd.DataFrame):
    plt.figure(figsize=(10, 10))
    sns.heatmap(tfidf_matrix, annot=True, cmap="YlGnBu", fmt=".2f", annot_kws={'rotation': 90})
    plt.xticks(rotation=90, ha="right")
    plt.tight_layout()
    plt.show()

    # CHANGE cmap to change color palette

In [ ]:
### ADJUSTING TO 20


# Prepare the TF-IDF matrix for visualization and EDA
tfidf_matrix = pd.DataFrame([happy_tf, unhappy_tf, younger_tf, older_tf], index=["Happy", "Unhappy", "Younger", "Older"])

# Get the top 10 words for each group
top_words_per_group = {}
for index, row in tfidf_matrix.iterrows():
    top_words = row.nlargest(20).index.tolist()
    top_words_per_group[index] = top_words

# Flatten the list of top words and get unique words
all_top_words = list(set([word for sublist in top_words_per_group.values() for word in sublist]))

# Filter the tfidf_matrix to include only these top words
tfidf_matrix_top20 = tfidf_matrix[all_top_words]

# Visualize the TF-IDF matrix
visualize_tfidf_rotated(tfidf_matrix_top20)

In [ ]:
# If we need 20 and it makes it squished, here's a version that turns the scores sideways

# Visualization function
def visualize_tfidf(tfidf_matrix: pd.DataFrame):
    plt.figure(figsize=(10, 10))
    sns.heatmap(tfidf_matrix, annot=True, cmap="YlGnBu")
    plt.xticks(rotation=90, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
### PCA and TSNE from google results

# 1. Sample Corpus
corpus = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?'
]

# 2. Vectorize
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# 3. Reduce Dimensionality to 2D
# TruncatedSVD is suitable for sparse TF-IDF matrices
svd = TruncatedSVD(n_components=2)
reduced_data = svd.fit_transform(tfidf_matrix)

# 4. Plot (using matplotlib)
plt.figure(figsize=(8, 6))
# Create a scatter plot of the documents in 2D space
plt.scatter(reduced_data[:, 0], reduced_data[:, 1])

# Annotate points with document labels
for i, doc in enumerate(corpus):
    plt.annotate(f'Doc {i+1}', (reduced_data[i, 0], reduced_data[i, 1]), textcoords="offset points", xytext=(0,5), ha='center')

plt.title('Document Visualization via TruncatedSVD')
plt.xlabel('SVD Component 1')
plt.ylabel('SVD Component 2')
plt.show()


In [ ]:
# HEAT MAP from google results


# Using the same vectorizer and tfidf_matrix from above...

# Get feature names (words) and document labels
feature_names = vectorizer.get_feature_names_out()
doc_labels = [f'Doc {i+1}' for i in range(len(corpus))]

# Convert sparse matrix to dense array
tfidf_array = tfidf_matrix.toarray()

# Plot the heatmap
plt.figure(figsize=(10, 5))
plt.imshow(tfidf_array, interpolation='nearest', cmap=plt.cm.YlGnBu)
plt.colorbar(label='TF-IDF Score')

# Set ticks and labels
plt.xticks(range(len(feature_names)), feature_names, rotation=45, ha='right')
plt.yticks(range(len(doc_labels)), doc_labels)

plt.xlabel('Terms')
plt.ylabel('Documents')
plt.title('TF-IDF Heatmap (Document-Term Matrix)')
plt.tight_layout()
plt.show()


In [ ]:
# Bar chart version from google

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import numpy as np

# Sample Corpus (replace with your actual data)
corpus = [
    "Data Science is the sexiest job of the 21st century",
    "machine learning is the key for data science",
    "TF-IDF is a useful technique for text mining",
    "Bar charts are a great way to visualize data"
]

# 1. Initialize TfidfVectorizer
# You can customize stop_words, ngram_range, etc.
vectorizer = TfidfVectorizer(stop_words='english')

# 2. Fit and transform the corpus to get the TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(corpus)

# 3. Get feature names (words) and TF-IDF scores for a specific document
# Here we'll focus on the first document (index 0)
document_index = 0
feature_names = vectorizer.get_feature_names_out()
tfidf_scores = tfidf_matrix[document_index].toarray().flatten()

# 4. Create a Pandas DataFrame for easier sorting and plotting
df = pd.DataFrame({'term': feature_names, 'tfidf_score': tfidf_scores})

# Filter out terms with zero scores (optional)
df = df[df['tfidf_score'] > 0]

# Sort the DataFrame by TF-IDF score in descending order
df = df.sort_values(by='tfidf_score', ascending=False)

# Select the top N terms to plot (e.g., top 10)
top_n = 10
df_top_n = df.head(top_n)

# 5. Plot the results using Matplotlib
plt.figure(figsize=(10, 6))
plt.barh(y=df_top_n['term'], width=df_top_n['tfidf_score'], color='skyblue')
plt.xlabel("TF-IDF Score", fontsize=12, fontweight="bold")
plt.ylabel("Term", fontsize=12, fontweight="bold")
plt.title(f"Top {top_n} TF-IDF Terms for Document {document_index + 1}", fontsize=14, fontweight="bold", pad=20)
plt.gca().invert_yaxis() # Invert y-axis to have the highest score at the top
plt.grid(axis="x", alpha=0.3, linestyle="--")

# Add value labels to the bars
for index, value in enumerate(df_top_n['tfidf_score']):
    plt.text(value, index, f'{value:.2f}', va='center', ha='left', fontsize=10)

plt.tight_layout()
plt.show()


# LDA Topic Allocation Modeling

In [ ]:
# LDA Topic Allocation

# CountVectorizer for LDA - Fit on original training data
count_vect = CountVectorizer( # keeping stop words
    max_df=0.95,
    min_df=2,
    ngram_range=(1, 2)
)
X_train_counts = count_vect.fit_transform(X_train)
X_val_counts   = count_vect.transform(X_val)

# LDA with n_topics - Fit on ORIGINAL training data
n_topics = 35
lda = LatentDirichletAllocation(
    n_components=n_topics,
    learning_method="batch",
    max_iter=100,
    random_state=42
)
train_doc_topic = lda.fit_transform(X_train_counts)
val_doc_topic   = lda.transform(X_val_counts)


# Topic introspection
def _show_top_words(model, feature_names, n_top=10):
    for t_idx, comp in enumerate(model.components_):
        top_ids = np.argsort(comp)[-n_top:][::-1]
        words = [feature_names[i] for i in top_ids]
        print(f"Topic {t_idx}: {', '.join(words)}")

print("\n=== LDA Topics (top words) ===")
_show_top_words(lda, count_vect.get_feature_names_out(), n_top=10)